A análise se baseará em order_reviews para descobrir os motivos principais das notas abaixo de 3 estrelas

In [1]:
import matplotlib as mtp
import pandas as pd
import numpy as np

order_payments = pd.read_csv("../archive/olist_order_payments_dataset.csv") #
order_itens = pd.read_csv("../archive/olist_order_items_dataset.csv") #
order_reviews = pd.read_csv("../archive/olist_order_reviews_dataset.csv") #
sellers = pd.read_csv("../archive/olist_sellers_dataset.csv") # 
products =  pd.read_csv("../archive/olist_products_dataset.csv") #
orders =  pd.read_csv("../archive/olist_orders_dataset.csv") #

order_payments.drop_duplicates()
order_itens.drop_duplicates()
order_reviews.drop_duplicates()
sellers.drop_duplicates()
products.drop_duplicates()
orders.drop_duplicates()

order_payments = order_payments.replace(r'^\s*$', np.nan, regex=True)
order_itens = order_itens.replace(r'^\s*$', np.nan, regex=True)
order_reviews = order_reviews.replace(r'^\s*$', np.nan, regex=True)
sellers = sellers.replace(r'^\s*$', np.nan, regex=True)
products = products.replace(r'^\s*$', np.nan, regex=True)
orders = orders.replace(r'^\s*$', np.nan, regex=True)

order_reviews["review_comment_message"] = order_reviews["review_comment_message"].replace(
    r"^\s*$", np.nan, regex=True
)
assert not order_reviews["review_comment_message"].dropna().str.fullmatch(r"\s*").any()

In [2]:

order_reviews_menor_q_3 = order_reviews.loc[order_reviews['review_score'] < 3]
print(f"{len(order_reviews_menor_q_3)} reviews com nota menor que 3")

distribuicao_order_reviews_menor_q_3 =(
    order_reviews_menor_q_3.groupby('review_score')
    .agg(
        qtd=('review_score','count')
    )
)
# porcentagem de reviews com nota menor que 3
media_order_review_menor_que_3 = len(order_reviews.query("review_score <  3")) / len(order_reviews) * 100
print(f"{media_order_review_menor_que_3:.1f}% menor que 3")

# média geral das reviews
media_order_reviews_geral = float(order_reviews["review_score"].mean())
print(f"{media_order_reviews_geral:.2f}/5 - média geral das reviews")
# truncado = float(str(media)[: str(media).find(".") + 3])

condicao = (
    (order_reviews["review_score"] < 3)
    & order_reviews["review_comment_message"].notna()
    & order_reviews["review_comment_message"].str.strip().ne("")
)

# below4comments = condicao.mean()
# below4comments

order_reviews_complaints = pd.DataFrame(order_reviews.loc[condicao])
order_reviews_complaints
order_reviews_complaints = order_reviews_complaints.drop_duplicates()
order_reviews_complaints.to_csv("created-tables/olist_order_reviews_complaints.csv")

distribuicao_order_reviews_menor_q_3

14575 reviews com nota menor que 3
14.7% menor que 3
4.09/5 - média geral das reviews


,qtd
review_score,
1,11424
2,3151


Gerando um novo csv com os dados limpos que precisamos da tabela de reviews

In [3]:
order_reviews_complaints_full_data = (
    order_reviews_complaints[['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']]
    .merge(
        order_itens[['order_id','order_item_id','product_id','seller_id','price','freight_value']],
        on='order_id',
        how='left'
    ).merge(
        sellers[['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']],
        on='seller_id',
        how='left'
    ).merge(
        products[['product_id','product_category_name']],
        on='product_id',
        how='left'
    ).merge(
        order_payments,
        on='order_id',
        how='left'
    ).merge(
        orders[['order_id','order_status','order_purchase_timestamp','order_delivered_customer_date','order_estimated_delivery_date']],
        on='order_id',
        how='left',
    )
)

order_reviews_complaints_full_data = order_reviews_complaints_full_data.drop_duplicates()
order_reviews_complaints_full_data['seller_zip_code_prefix'] = order_reviews_complaints_full_data['seller_zip_code_prefix'].astype(str).str.strip()  
order_reviews_complaints_full_data['order_item_id'] = order_reviews_complaints_full_data['order_item_id'].astype('Int64')
order_reviews_complaints_full_data['payment_sequential'] = order_reviews_complaints_full_data['payment_sequential'].astype('Int64')
order_reviews_complaints_full_data['payment_installments'] = order_reviews_complaints_full_data['payment_installments'].astype('Int64')
order_reviews_complaints_full_data.dropna(how="all")

order_reviews_complaints_full_data.shape
order_reviews_complaints_full_data.to_csv("created-tables/order_review_complaints_full_data.csv")

Gerando uma nova tabela unindo dados de outras para uma melhor análise

In [4]:
# !!! PROBLEMA: Estou encontrando muitas reviews e orders repetidas.

# pd.unique(order_reviews_complaints_full_data['review_id'])
order_reviews_complaints_analise_repetidos = order_reviews_complaints_full_data[
    [
        'review_id',
        'order_id',
        'price',
        'freight_value',
        'payment_value',
        'payment_type',
        'review_comment_message',
        'review_score',
        'order_purchase_timestamp',
        'order_delivered_customer_date',
        'order_estimated_delivery_date'
    ]
].dropna(subset='price')
order_reviews_complaints_analise_repetidos.to_csv('created-tables//order_reviews_complaints_analise_repetidos.csv')

order_reviews_complaints_analise_nao_repetidos = order_reviews_complaints_analise_repetidos.drop_duplicates(subset='order_id')
order_reviews_complaints_analise_nao_repetidos.to_csv('created-tables/order_reviews_complaints_analise_nao_repetidos.csv')

Porcentagem de entregas dentro do prazo

In [6]:
# percebi que os pedidos repetidos eram mais produtos na msm order e por isso serão desconsiderados para analise
porcentagem_de_entrega_dentro_do_tempo_notas_ruins = float((order_reviews_complaints_analise_nao_repetidos['order_delivered_customer_date'] < order_reviews_complaints_analise_nao_repetidos['order_estimated_delivery_date']).mean() * 100)

order_reviews_complaints_full_data_all = (
    order_reviews[['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']]
    .merge(
        order_itens[['order_id','order_item_id','product_id','seller_id','price','freight_value']],
        on='order_id',
        how='left'
    ).merge(
        sellers[['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']],
        on='seller_id',
        how='left'
    ).merge(
        products[['product_id','product_category_name']],
        on='product_id',
        how='left'
    ).merge(
        order_payments,
        on='order_id',
        how='left'
    ).merge(
        orders[['order_id','order_status','order_purchase_timestamp','order_delivered_customer_date','order_estimated_delivery_date']],
        on='order_id',
        how='left',
    )
)

order_reviews_complaints_full_data_all = order_reviews_complaints_full_data_all.drop_duplicates()
order_reviews_complaints_full_data_all['seller_zip_code_prefix'] = order_reviews_complaints_full_data_all['seller_zip_code_prefix'].astype(str).str.strip()  
order_reviews_complaints_full_data_all['order_item_id'] = order_reviews_complaints_full_data_all['order_item_id'].astype('Int64')
order_reviews_complaints_full_data_all['payment_sequential'] = order_reviews_complaints_full_data_all['payment_sequential'].astype('Int64')
order_reviews_complaints_full_data_all['payment_installments'] = order_reviews_complaints_full_data_all['payment_installments'].astype('Int64')
order_reviews_complaints_full_data_all.dropna(how="all")

order_reviews_complaints_full_data_all.shape
order_reviews_complaints_full_data_all.to_csv("created-tables/all_order_reviews_full_data.csv")

porcentagem_de_entrega_dentro_do_tempo_notas_boas = float((order_reviews_complaints_full_data_all['order_delivered_customer_date'] < order_reviews_complaints_full_data_all['order_estimated_delivery_date']).mean() * 100)

print(f"notas ruins: {porcentagem_de_entrega_dentro_do_tempo_notas_ruins:.2f} \nnotas boas: {porcentagem_de_entrega_dentro_do_tempo_notas_boas:.2f}")

notas ruins: 61.46 
notas boas: 89.72
